In [ ]:
import pandas as pd
from sklearn.preprocessing import minmax_scale

In [46]:
df = pd.read_csv("final_transformations/F2_SAMPLE_DATA_WITH_TIMESTAMP.csv")
# df_mae_p = pd.read_csv("mae_producto_20rows.csv")

"""
Columns
= alternative
== same
1. Tipo subestrategia == DES_TIPO_SUBESTRATEGIA
2. Tipo grupo == DES_TIPO_GRUPO
3. Indicator padre == ES_PADRE
4. Indicator gratis = ES_GRATIS
5. Factor of repetition == FACTOR_REPETICION

"""

'\nColumns\n= alternative\n== same\n1. Tipo subestrategia == DES_TIPO_SUBESTRATEGIA\n2. Tipo grupo == DES_TIPO_GRUPO\n3. Indicator padre == ES_PADRE\n4. Indicator gratis = ES_GRATIS\n5. Factor of repetition == FACTOR_REPETICION\n\n'

In [47]:
unique_occurrences = df['ANIOCAMPANA'].value_counts()

print("Unique occurrences:")
print(unique_occurrences)

Unique occurrences:
ANIOCAMPANA
202305    298
202303    242
202302    231
202304    218
202301    187
202307    121
202306    104
202308     59
202309     34
202313     25
202312     19
202311     17
202310     17
202314     16
Name: count, dtype: int64


In [49]:
# Divide the DataFrame based on unique values in the 'Category' column
dfs = {category: df_subset for category, df_subset in df.groupby('ANIOCAMPANA')}
i=1
# Dynamically create variable names for each sub_df
for category, sub_df in dfs.items():
    # Create a unique name for each sub_df (based on category)
    variable_name = f"df_{i}"
    
    # Assign the sub_df to the unique variable name
    globals()[variable_name] = sub_df
    
    # Optionally, print the name and content to verify
    print(f"Created variable: {variable_name}")
    i+=1

Created variable: df_1
Created variable: df_2
Created variable: df_3
Created variable: df_4
Created variable: df_5
Created variable: df_6
Created variable: df_7
Created variable: df_8
Created variable: df_9
Created variable: df_10
Created variable: df_11
Created variable: df_12
Created variable: df_13
Created variable: df_14


In [101]:
# Assuming `dfs` is a dictionary containing the 14 DataFrames
# Define the function for recency calculation
def cal_recency(ref_date, last_purchase_date):
    last_purchase_date = pd.Timestamp(last_purchase_date)
    days_difference = (ref_date - last_purchase_date).days
    recency = 1 / (days_difference + 1)  
    return recency

# Reference date for recency calculation
reference_date = pd.Timestamp('2024-12-29')  # You can replace this with the current date if needed

all_dfs = []
# Loop through each DataFrame, apply transformations, and save them
for i, (category, df) in enumerate(dfs.items(), 1):
    # Step 1: Filter the DataFrame
    df_filtered = df[df["CODCUC"] != "XXXXXXXXX"]

    # Step 2: Create Composite_key column
    df_filtered["Composite_key"] = df_filtered[[
        "DES_TIPO_SUBESTRATEGIA", "DES_TIPO_GRUPO", "CODCUC", "ES_PADRE", 
        "ES_GRATIS", "FACTOR_REPETICION"
    ]].astype(str).agg('|'.join, axis=1)

    # Step 3: Drop the "COMPOSITE_PRIMARY_KEY" column
    df_filtered = df_filtered.drop("COMPOSITE_PRIMARY_KEY", axis=1, errors='ignore')

    # Step 4: Calculate recency
    recency_values = [cal_recency(reference_date, date) for date in df_filtered['FECHAPROCESO']]
    df_filtered["recency"] = recency_values

    # Step 5: Concatenate the grouped data
    concatenateds = df_filtered.groupby("ID_OFERTA")[["Composite_key", "CODEBELISTA", "recency"]].agg(
        {
            'CODEBELISTA': lambda x: x.dropna().tolist(),
            'Composite_key': lambda x: '|'.join(x),
            'recency': 'mean'  # Aggregating recency by mean
        }
    ).reset_index()

    # Step 6: Expand the CODEBELISTA column into separate rows
    expanded_df = concatenateds.explode('CODEBELISTA')

    # Step 7: Group by CODEBELISTA and Composite_key, aggregate ID_OFERTA into a list
    grouped_df = expanded_df.groupby(['CODEBELISTA', 'Composite_key']).agg({
        'ID_OFERTA': lambda x: x.tolist(),
        'recency': 'mean'
    }).reset_index()

    # Step 8: Add count column
    grouped_df['count'] = expanded_df.groupby(['CODEBELISTA', 'Composite_key']).size().values

    # Step 9: Normalize the count and recency columns
    grouped_df['normalized_count'] = minmax_scale(grouped_df['count'])
    grouped_df['normalized_recency'] = minmax_scale(grouped_df['recency'])

    # Step 10: Calculate the score
    def score_calculation(recency, frequency, weight1=0.3, weight2=0.7):
        return weight1 * recency + frequency * weight2

    grouped_df['score'] = grouped_df.apply(lambda row: score_calculation(row['normalized_recency'], row['normalized_count']), axis=1)

    # Step 11: Sort by score in descending order
    sorted_df = grouped_df.sort_values('score', ascending=False)
    
    all_dfs.append(sorted_df)
    # Step 12: Save the result to a CSV file
    file_name = f"{category}.csv"
    sorted_df.to_csv(file_name, index=False)

    print(f"Saved {file_name}")

final_df = pd.concat(all_dfs, ignore_index=True)

Saved 202301.csv
Saved 202302.csv
Saved 202303.csv
Saved 202304.csv
Saved 202305.csv
Saved 202306.csv
Saved 202307.csv
Saved 202308.csv
Saved 202309.csv
Saved 202310.csv
Saved 202311.csv
Saved 202312.csv
Saved 202313.csv
Saved 202314.csv


/var/folders/bh/74g24pss1j3_fnkxfr5pf0br0000gn/T/ipykernel_29142/3047100830.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["Composite_key"] = df_filtered[[
/var/folders/bh/74g24pss1j3_fnkxfr5pf0br0000gn/T/ipykernel_29142/3047100830.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["Composite_key"] = df_filtered[[
/var/folders/bh/74g24pss1j3_fnkxfr5pf0br0000gn/T/ipykernel_29142/3047100830.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

Filtering out duplicate offers from CAT, REV, DIG

In [93]:
df_future = pd.read_csv("../src/data/files/F2_Future_Campaign_Data_PE.csv")

In [94]:
# For future campaigns
df_future_filtered = df_future[df_future["CODCUC"] != "XXXXXXXXX"]

print(df_future_filtered.shape)

(566995, 26)


In [95]:
df_future_filtered["Composite_key"] = df_future_filtered[["DES_TIPO_SUBESTRATEGIA","DES_TIPO_GRUPO","CODCUC","ES_PADRE","ES_GRATIS","FACTOR_REPETICION"]].astype(str).agg('|'.join, axis=1)

/var/folders/bh/74g24pss1j3_fnkxfr5pf0br0000gn/T/ipykernel_29142/2214267813.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_future_filtered["Composite_key"] = df_future_filtered[["DES_TIPO_SUBESTRATEGIA","DES_TIPO_GRUPO","CODCUC","ES_PADRE","ES_GRATIS","FACTOR_REPETICION"]].astype(str).agg('|'.join, axis=1)


In [96]:
# For future campaigns
df_future_filtered = df_future_filtered.drop("COMPOSITE_PRIMARY_KEY", axis=1)  # Remove "Column2"
print(df_future_filtered.shape)

(566995, 26)


In [97]:
unique_occurrences = df_future_filtered["COD_PERIODO"].value_counts()

print("Unique occurrences:")
print(unique_occurrences)

Unique occurrences:
COD_PERIODO
202501    300552
202502    180446
202503     28974
202504     27799
202505     23357
202506      5867
Name: count, dtype: int64


In [98]:
# Divide the DataFrame based on unique values in the 'COD_PERIODO' column
df_futures = {campaign_id: df_subset for campaign_id, df_subset in df_future_filtered.groupby('COD_PERIODO')}
i=1
# Dynamically create variable names for each sub_df
for campaign_id, sub_df in df_futures.items():
    # Create a unique name for each sub_df (based on category)
    variable_name = f"df_{i}"
    
    # Assign the sub_df to the unique variable name
    globals()[variable_name] = sub_df
    
    # Optionally, print the name and content to verify
    print(f"Created variable: {variable_name}")
    print(globals()[variable_name])
    print()
    i+=1


Created variable: df_1
       COD_PAIS  COD_PERIODO  COD_VENTA    COD_SAP  ID_OFERTA  COD_CATALOGO  \
36           PE       202501       4946  200060841       3317          35.0   
37           PE       202501       4946  200060841        999          47.0   
43           PE       202501     118426  210092515       2384          24.0   
44           PE       202501     118328  210092515       2383          24.0   
103          PE       202501     100236  200064341       6490          45.0   
...         ...          ...        ...        ...        ...           ...   
600554       PE       202501      37287  200113704       2653          24.0   
600557       PE       202501      37291  200113704       2651          24.0   
600559       PE       202501     118403  200113704       2335          24.0   
600563       PE       202501      37290  200113704       2654          24.0   
600564       PE       202501      37289  200113704       2652          24.0   

        ID_MACROESTRATEGIA  

In [102]:
# Assuming dfs contains the smaller DataFrames (subsets) from the previous steps
# Example of transformations on each sub_df
all_future_dfs = []
for i, (campaign_id, sub_df) in enumerate(df_futures.items(), 1):
    
    # Perform the groupby and aggregation
    sub_df_concatenated = sub_df.groupby("ID_OFERTA")[["Composite_key"]].agg(
        {'Composite_key': lambda x: '|'.join(x)}
    ).reset_index()
    
    all_future_dfs.append(sub_df_concatenated)
    # Save the result to a CSV file with a dynamic name
    file_name = f"{campaign_id}.csv"
    sub_df_concatenated.to_csv(file_name, index=False)
    
    print(f"Saved {file_name}")
final_future_df = pd.concat(all_future_dfs, ignore_index=True)

Saved 202501.csv
Saved 202502.csv
Saved 202503.csv
Saved 202504.csv
Saved 202505.csv
Saved 202506.csv


In [103]:
duplicate_offers_count = final_future_df['Composite_key'].isin(final_df['Composite_key']).sum()
print(f"Number of matching Composite_key values: {duplicate_offers_count}")

Number of matching Composite_key values: 0
